# Gemini + Places Restaurant Recommender

This Colab notebook uses:
- **Gemini** for slot filling, follow-up questions, confirmations, and the final recommendation paragraph
- **Google Places API (New)** for live restaurant discovery and place details

The idea is simple: Gemini handles the conversation, while Places handles the facts.


## Setup

Before running the notebook, make sure:
- your `GOOGLE_MAPS_API_KEY` is enabled for **Places API (New)**
- your Gemini key works in Google AI Studio
- billing is enabled on the Google Cloud project for Places
- your key restrictions allow calls to `places.googleapis.com`


In [ ]:
%pip install -q -U google-genai requests folium


In [ ]:
import json
import math
import os
import re
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from getpass import getpass
from typing import Any, Dict, List, Optional
import folium
import requests
from google import genai
from IPython.display import display


In [ ]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or getpass("Enter GEMINI_API_KEY: ")
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY") or getpass("Enter GOOGLE_MAPS_API_KEY: ")

GEMINI_MODEL = "gemini-2.0-flash"
PLACES_TEXT_SEARCH_URL = "https://places.googleapis.com/v1/places:searchText"
PLACES_DETAILS_URL = "https://places.googleapis.com/v1/places/{place_id}"

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini and Places keys captured.")


In [ ]:
def normalize_text(text: str) -> str:
    text = str(text or "").strip().lower()
    text = re.sub(r"[^a-z0-9\s.,:/-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def extract_json_object(text: str) -> Dict[str, Any]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError(f"Could not find JSON in Gemini response: {text}")
    return json.loads(text[start:end + 1])


def haversine_miles(lat1: float, lng1: float, lat2: float, lng2: float) -> float:
    radius_miles = 3958.8
    d_lat = math.radians(lat2 - lat1)
    d_lng = math.radians(lng2 - lng1)
    a = (
        math.sin(d_lat / 2) ** 2
        + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(d_lng / 2) ** 2
    )
    return 2 * radius_miles * math.asin(math.sqrt(a))


def places_text_search(text_query: str, field_mask: str, max_result_count: int = 10, location_bias: Optional[dict] = None) -> List[dict]:
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_MAPS_API_KEY,
        "X-Goog-FieldMask": field_mask,
    }
    payload = {
        "textQuery": text_query,
        "maxResultCount": max_result_count,
    }
    if location_bias:
        payload["locationBias"] = location_bias
    response = requests.post(PLACES_TEXT_SEARCH_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return response.json().get("places", [])


def place_details(place_id: str, field_mask: str) -> dict:
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_MAPS_API_KEY,
        "X-Goog-FieldMask": field_mask,
    }
    response = requests.get(PLACES_DETAILS_URL.format(place_id=place_id), headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()




def _build_coordinate_result(lat: float, lng: float) -> Optional[Dict[str, float]]:
    if not (-90 <= lat <= 90 and -180 <= lng <= 180):
        return None
    return {
        "label": f"{lat:.6f}, {lng:.6f}",
        "address": f"Coordinates: {lat:.6f}, {lng:.6f}",
        "lat": lat,
        "lng": lng,
    }


def _dms_to_decimal(degrees: float, minutes: float, seconds: float, hemisphere: str) -> float:
    value = abs(degrees) + minutes / 60.0 + seconds / 3600.0
    if hemisphere.upper() in {"S", "W"}:
        value *= -1
    return value


def parse_coordinate_pair(text: str) -> Optional[Dict[str, float]]:
    cleaned = str(text or "").strip()

    decimal_match = re.search(r"(-?\d{1,3}(?:\.\d+)?)\s*,\s*(-?\d{1,3}(?:\.\d+)?)", cleaned)
    if decimal_match:
        lat = float(decimal_match.group(1))
        lng = float(decimal_match.group(2))
        return _build_coordinate_result(lat, lng)

    dms_pattern = re.compile(
        r"(\d{1,3})[^0-9A-Za-z]+(\d{1,2})[^0-9A-Za-z]+(\d{1,2}(?:\.\d+)?)[^0-9A-Za-z]*([NS])\D+"
        r"(\d{1,3})[^0-9A-Za-z]+(\d{1,2})[^0-9A-Za-z]+(\d{1,2}(?:\.\d+)?)[^0-9A-Za-z]*([EW])",
        re.IGNORECASE,
    )
    dms_match = dms_pattern.search(cleaned)
    if dms_match:
        lat = _dms_to_decimal(float(dms_match.group(1)), float(dms_match.group(2)), float(dms_match.group(3)), dms_match.group(4))
        lng = _dms_to_decimal(float(dms_match.group(5)), float(dms_match.group(6)), float(dms_match.group(7)), dms_match.group(8))
        return _build_coordinate_result(lat, lng)

    return None

def geocode_location(location_text: str) -> dict:
    coordinates = parse_coordinate_pair(location_text)
    if coordinates:
        return coordinates

    places = places_text_search(
        text_query=location_text,
        field_mask="places.id,places.displayName,places.formattedAddress,places.location",
        max_result_count=1,
    )
    if not places:
        raise ValueError(f"Could not resolve location: {location_text}")

    place = places[0]
    return {
        "label": place.get("displayName", {}).get("text", location_text),
        "address": place.get("formattedAddress", location_text),
        "lat": place["location"]["latitude"],
        "lng": place["location"]["longitude"],
    }


def get_browser_location_colab() -> dict:
    from google.colab import output

    js = """
    async function getLocation() {
      return await new Promise((resolve, reject) => {
        navigator.geolocation.getCurrentPosition(
          (position) => resolve({lat: position.coords.latitude, lng: position.coords.longitude}),
          (error) => reject(error.message),
          {enableHighAccuracy: true, timeout: 10000}
        );
      });
    }
    """
    coords = output.eval_js(js + "getLocation()")
    return {
        "label": "Current location",
        "address": "Current browser location",
        "lat": float(coords["lat"]),
        "lng": float(coords["lng"]),
    }


def parse_review_timestamp(value: Optional[str]) -> Optional[datetime]:
    if not value:
        return None
    try:
        normalized = value.replace("Z", "+00:00")
        return datetime.fromisoformat(normalized)
    except ValueError:
        return None


def review_recency_weight(review: Dict[str, Any], now: Optional[datetime] = None) -> float:
    now = now or datetime.now(timezone.utc)
    published = parse_review_timestamp(review.get("publishTime"))
    if not published:
        return 0.0

    if published.tzinfo is None:
        published = published.replace(tzinfo=timezone.utc)

    age = now - published
    if age > timedelta(days=730):
        return 0.0

    # Last 12 months get the strongest weight. Months 12-24 still count,
    # but they are penalized more heavily so fresher reviews dominate.
    if age <= timedelta(days=365):
        freshness = 1.0 - (age.days / 365.0) * 0.35
    else:
        extra_days = age.days - 365
        freshness = max(0.12, 0.65 - (extra_days / 365.0) * 0.5)

    rating = float(review.get("rating", 0.0) or 0.0)
    sentiment = max(0.0, rating / 5.0)
    return freshness * sentiment


def summarize_recent_reviews(reviews: List[Dict[str, Any]]) -> Dict[str, Any]:
    now = datetime.now(timezone.utc)
    recent_reviews = []
    weighted_sum = 0.0
    total_weight = 0.0

    for review in reviews or []:
        weight = review_recency_weight(review, now=now)
        if weight <= 0:
            continue
        rating = float(review.get("rating", 0.0) or 0.0)
        weighted_sum += rating * weight
        total_weight += weight
        recent_reviews.append({
            "rating": rating,
            "publishTime": review.get("publishTime"),
            "relativePublishTimeDescription": review.get("relativePublishTimeDescription"),
            "text": (review.get("text") or {}).get("text"),
            "weight": round(weight, 4),
        })

    recent_reviews.sort(key=lambda item: item.get("publishTime") or "", reverse=True)
    return {
        "recent_reviews": recent_reviews,
        "recent_review_count": len(recent_reviews),
        "recency_weighted_rating": round(weighted_sum / total_weight, 2) if total_weight > 0 else None,
    }


DISH_TO_CUISINE = {
    "pizza": "italian",
    "margherita pizza": "italian",
    "chicago pizza": "italian",
    "hawaiian pizza": "italian",
    "pasta": "italian",
    "dosa": "south indian",
    "idli": "south indian",
    "vada": "south indian",
    "biryani": "indian",
    "sushi": "japanese",
    "ramen": "japanese",
    "taco": "mexican",
    "tacos": "mexican",
    "burrito": "mexican",
    "kimchi": "korean",
    "bibimbap": "korean",
    "pad thai": "thai",
    "pho": "vietnamese",
    "coffee": "coffee",
}

VAGUE_FOOD_INTENTS = {
    "breakfast", "brunch", "lunch", "dinner", "snack", "dessert", "sweets", "ice cream", "food", "eat"
}


def infer_cuisine_from_dish(dish: Optional[str]) -> Optional[str]:
    cleaned = normalize_text(dish or "")
    if not cleaned:
        return None
    for key, cuisine in sorted(DISH_TO_CUISINE.items(), key=lambda item: len(item[0]), reverse=True):
        if key in cleaned:
            return cuisine
    return None


def is_vague_food_request(dish: Optional[str]) -> bool:
    cleaned = normalize_text(dish or "")
    return cleaned in VAGUE_FOOD_INTENTS



def build_results_map(user_location: Dict[str, Any], results: List[Dict[str, Any]]) -> folium.Map:
    fmap = folium.Map(location=[user_location["lat"], user_location["lng"]], zoom_start=14)

    folium.Marker(
        [user_location["lat"], user_location["lng"]],
        tooltip="Your location",
        popup=f"Your location: {user_location.get('label', 'Current location')}",
        icon=folium.Icon(color="blue", icon="user", prefix="fa"),
    ).add_to(fmap)

    valid_results = [place for place in results[:5] if place.get("lat") is not None and place.get("lng") is not None]

    for idx, place in enumerate(valid_results, start=1):
        lat = place.get("lat")
        lng = place.get("lng")

        popup_lines = [f"<b>{idx}. {place.get('name', 'Place')}</b>"]
        if place.get("address"):
            popup_lines.append(place["address"])
        if place.get("recency_weighted_rating") is not None:
            popup_lines.append(f"Recent review score: {place['recency_weighted_rating']}")
        elif place.get("rating") is not None:
            popup_lines.append(f"Rating: {place['rating']}")
        if place.get("distance_miles") is not None:
            popup_lines.append(f"Distance from your location: {place['distance_miles']} miles")
        if place.get("estimated_travel_minutes") is not None:
            popup_lines.append(f"Estimated travel time: {place['estimated_travel_minutes']} minutes")
        if place.get("unmet_criteria"):
            popup_lines.append("Does not fully follow: " + ", ".join(place["unmet_criteria"]))
        if place.get("directions_url"):
            popup_lines.append(f"<a href='{place['directions_url']}' target='_blank'>Get directions</a>")
        elif place.get("google_maps_url"):
            popup_lines.append(f"<a href='{place['google_maps_url']}' target='_blank'>Open in Google Maps</a>")

        tooltip = f"{idx}. {place.get('name', 'Place')}"
        if place.get("distance_miles") is not None:
            tooltip += f" | {place['distance_miles']} miles"

        folium.Marker(
            [lat, lng],
            tooltip=tooltip,
            popup=folium.Popup("<br>".join(popup_lines), max_width=320),
            icon=folium.Icon(color="red", icon="cutlery", prefix="fa"),
        ).add_to(fmap)

        folium.PolyLine(
            locations=[[user_location["lat"], user_location["lng"]], [lat, lng]],
            color="#4f83ff",
            weight=3,
            opacity=0.55,
        ).add_to(fmap)

        if place.get("distance_miles") is not None:
            midpoint = [
                (user_location["lat"] + lat) / 2.0,
                (user_location["lng"] + lng) / 2.0,
            ]
            folium.Marker(
                midpoint,
                tooltip=f"Distance to {place.get('name', 'Place')}: {place['distance_miles']} miles",
                icon=folium.DivIcon(
                    html=(
                        "<div style='background:#4f83ff;color:white;padding:4px 8px;border-radius:999px;"
                        "font-size:11px;font-weight:600;white-space:nowrap;'>"
                        f"{place['distance_miles']} mi"
                        "</div>"
                    )
                ),
            ).add_to(fmap)

    return fmap


def infer_travel_preferences(text: str) -> Dict[str, Any]:
    cleaned = normalize_text(text)
    mode = None
    if "walk" in cleaned or "walking" in cleaned or "by foot" in cleaned:
        mode = "walk"
    elif "car" in cleaned or "drive" in cleaned or "driving" in cleaned:
        mode = "car"

    minutes = None
    match = re.search(r"(\d{1,3})\s*(minute|min|minutes)", cleaned)
    if match:
        minutes = int(match.group(1))
    elif re.fullmatch(r"\d{1,3}", cleaned):
        minutes = int(cleaned)

    radius_meters = None
    if minutes is not None:
        if mode == "walk":
            radius_meters = max(500, min(minutes * 80, 5000))
        elif mode == "car":
            radius_meters = max(1000, min(minutes * 700, 30000))
        else:
            radius_meters = max(1000, min(minutes * 250, 15000))

    return {
        "travel_mode": mode,
        "travel_minutes": minutes,
        "search_radius_meters": radius_meters,
    }


def travel_preference_label(state: Dict[str, Any]) -> str:
    minutes = state.get("travel_minutes")
    mode = state.get("travel_mode")
    if minutes and mode:
        return f"{minutes} minutes by {mode}"
    if minutes:
        return f"{minutes} minutes"
    if mode:
        return mode
    return "not specified"


def estimate_travel_minutes(distance_miles: Optional[float], travel_mode: Optional[str]) -> Optional[int]:
    if distance_miles is None:
        return None
    if travel_mode == "walk":
        return max(1, int(round((distance_miles / 3.0) * 60)))
    if travel_mode == "car":
        return max(1, int(round((distance_miles / 20.0) * 60)))
    return max(1, int(round((distance_miles / 10.0) * 60)))


def parse_google_timestamp(value: Optional[str]) -> Optional[datetime]:
    if not value:
        return None
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return None


def will_close_before_arrival(next_close_time: Optional[str], travel_minutes: Optional[int]) -> bool:
    if not next_close_time or travel_minutes is None:
        return False
    close_dt = parse_google_timestamp(next_close_time)
    if not close_dt:
        return False
    return close_dt <= datetime.now(timezone.utc) + timedelta(minutes=travel_minutes)


def build_directions_url(user_location: Dict[str, Any], destination: Dict[str, Any]) -> str:
    origin = f"{user_location['lat']},{user_location['lng']}"
    dest = f"{destination['lat']},{destination['lng']}"
    return f"https://www.google.com/maps/dir/?api=1&origin={origin}&destination={dest}&travelmode=driving"


def unmet_criteria_for_place(place: Dict[str, Any], state: Dict[str, Any]) -> List[str]:
    unmet = []
    min_rating = state.get("min_rating")
    if min_rating is not None and place.get("rating") is not None and float(place["rating"]) < float(min_rating):
        unmet.append(f"rating is below your minimum of {min_rating}")

    preferred_minutes = state.get("travel_minutes")
    estimated_minutes = place.get("estimated_travel_minutes")
    mode = state.get("travel_mode") or "travel"
    if preferred_minutes and estimated_minutes and estimated_minutes > preferred_minutes:
        unmet.append(f"estimated {estimated_minutes} minutes by {mode}, which is above your preferred {preferred_minutes}")

    if state.get("when") == "now" and place.get("open_now") is False:
        unmet.append("not open right now")

    return unmet


In [ ]:
SLOT_SYSTEM_PROMPT = """
You are a restaurant recommendation conversation planner.

Your job is to read the conversation and update a restaurant preference state.
Return JSON only.

Rules:
- Use the conversation history and the current state.
- The only required slots are: when, cuisine, location, and travel preference.
- Extract or update these fields when possible: dish, cuisine, meal_type, place_type, ambience, min_rating, hunger_level, when, location_mode, manual_location, travel_mode, travel_minutes.
- Time means whether the user wants to eat now or later.
- If the user says something like 'I want to eat now' or 'right now', treat that as when='now'.
- If the user mentions a specific dish that strongly implies a cuisine, infer that cuisine instead of asking again. Examples: pizza -> italian, dosa -> south indian, sushi -> japanese, tacos -> mexican, coffee -> coffee or cafe.
- Only ask for cuisine when the user's first food intent is vague, such as breakfast, lunch, dinner, snack, dessert, or sweets.
- If the user has given time but not cuisine, ask about cuisine next only when the request is vague.
- If the user has given time and cuisine but not location, ask about location next.
- After time, cuisine, and location are available, ask how far they are willing to travel and whether that is by walk or by car.
- If the user has given cuisine and location but not time, ask whether they want to eat now or later.
- When cuisine is missing and must be asked, the next question should sound like 'What is your preference?' or 'What cuisine are you in the mood for?'
- If the user answers with only a number like 4 or 4.5 after a rating question, treat it as min_rating.
- If the user says manual, manually, or enter location, set location_mode to manual.
- If the user says use my location, current location, or yes you can use my location, set location_mode to current.
- If the assistant asked for a location and the user replies with a place name like Central Park or Empire State, set manual_location.
- If the user says south indian, preserve that as cuisine instead of only indian.
- If the user says anything is fine, store flexible for that slot if appropriate.
- Ask only one next question at a time.
- Keep the next question short and natural.
- Mark ready_for_search true only when time, cuisine, location, and travel preference are all available.
- Do not require ambience, place type, meal type, dish, or min_rating before search.

Return this exact shape:
{
  "state": {
    "dish": null,
    "cuisine": null,
    "meal_type": null,
    "place_type": null,
    "ambience": null,
    "min_rating": null,
    "hunger_level": null,
    "when": null,
    "location_mode": null,
    "manual_location": null,
    "travel_mode": null,
    "travel_minutes": null
  },
  "next_question": null,
  "confirmation": null,
  "ready_for_search": false
}
"""


@dataclass
class GeminiPlacesBot:
    state: Dict[str, Any] = field(default_factory=lambda: {
        "dish": None,
        "cuisine": None,
        "meal_type": None,
        "place_type": None,
        "ambience": None,
        "min_rating": None,
        "hunger_level": None,
        "when": None,
        "location_mode": None,
        "manual_location": None,
        "travel_mode": None,
        "travel_minutes": None,
        "search_radius_meters": None,
        "user_location": None,
    })
    history: List[Dict[str, str]] = field(default_factory=list)
    last_results: List[Dict[str, Any]] = field(default_factory=list)

    def _append(self, role: str, text: str) -> None:
        self.history.append({"role": role, "text": text})

    def _merge_state(self, updates: Dict[str, Any]) -> None:
        for key, value in updates.items():
            if key not in self.state:
                continue
            if value is None:
                continue
            if isinstance(value, str) and not value.strip():
                continue
            self.state[key] = value

    def _apply_local_food_inference(self) -> None:
        dish = self.state.get("dish")
        cuisine = self.state.get("cuisine")
        inferred = infer_cuisine_from_dish(dish)
        if inferred and not cuisine:
            self.state["cuisine"] = inferred

    def _apply_local_travel_inference(self, user_message: str) -> None:
        inferred = infer_travel_preferences(user_message)
        if inferred.get("travel_mode") and not self.state.get("travel_mode"):
            self.state["travel_mode"] = inferred["travel_mode"]
        if inferred.get("travel_minutes") and not self.state.get("travel_minutes"):
            self.state["travel_minutes"] = inferred["travel_minutes"]
        if inferred.get("search_radius_meters"):
            self.state["search_radius_meters"] = inferred["search_radius_meters"]

    def _apply_local_location_fallbacks(self, user_message: str) -> None:
        coordinates = parse_coordinate_pair(user_message)
        if coordinates:
            self.state["location_mode"] = "manual"
            self.state["manual_location"] = user_message.strip()
            self.state["user_location"] = coordinates
            return

        cleaned = normalize_text(user_message)
        if any(phrase in cleaned for phrase in ["manual", "enter location", "i will enter location manually"]):
            self.state["location_mode"] = "manual"
            self.state["user_location"] = None

    def _analyze_with_gemini(self) -> Dict[str, Any]:
        prompt = (
            SLOT_SYSTEM_PROMPT
            + "\nCurrent state:\n"
            + json.dumps({k: v for k, v in self.state.items() if k != 'user_location'}, ensure_ascii=False)
            + "\n\nConversation history:\n"
            + json.dumps(self.history[-12:], ensure_ascii=False)
        )
        response = gemini_client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        return extract_json_object(response.text)

    def _resolve_location(self) -> None:
        if self.state["user_location"]:
            return
        if self.state["location_mode"] == "current":
            self.state["user_location"] = get_browser_location_colab()
        elif self.state["location_mode"] == "manual" and self.state["manual_location"]:
            self.state["user_location"] = geocode_location(self.state["manual_location"])

    def _has_search_intent(self) -> bool:
        return bool(self.state["cuisine"] or self.state["dish"])

    def _build_search_query(self) -> str:
        parts = []
        if self.state["dish"]:
            parts.append(self.state["dish"])
        if self.state["cuisine"] and self.state["cuisine"] not in " ".join(parts):
            parts.append(self.state["cuisine"])
        if self.state["place_type"] and self.state["place_type"] != "flexible":
            parts.append(self.state["place_type"].replace("_", " "))
        elif self.state["meal_type"] == "coffee":
            parts.append("cafe")
        else:
            parts.append("restaurant")
        if self.state["ambience"] and self.state["ambience"] != "flexible":
            parts.append(self.state["ambience"])
        return " ".join(parts)

    def _fetch_recommendations(self, limit: int = 5) -> List[Dict[str, Any]]:
        self._resolve_location()
        if not self.state["user_location"]:
            raise ValueError("Location could not be resolved yet.")

        location_bias = {
            "circle": {
                "center": {
                    "latitude": self.state["user_location"]["lat"],
                    "longitude": self.state["user_location"]["lng"],
                },
                "radius": float(self.state.get("search_radius_meters") or 7000.0),
            }
        }

        places = places_text_search(
            text_query=self._build_search_query(),
            field_mask=(
                "places.id,places.displayName,places.formattedAddress,places.location,"
                "places.rating,places.userRatingCount,places.priceLevel,places.primaryTypeDisplayName"
            ),
            max_result_count=10,
            location_bias=location_bias,
        )

        results = []
        for place in places:
            rating = place.get("rating")
            min_rating = self.state.get("min_rating")
            if min_rating is not None and (rating is None or float(rating) < float(min_rating)):
                continue

            details = place_details(
                place["id"],
                (
                    "id,displayName,formattedAddress,location,rating,userRatingCount,priceLevel,"
                    "primaryTypeDisplayName,currentOpeningHours,regularOpeningHours,reviews,reviewSummary,"
                    "editorialSummary,takeout,delivery,dineIn,servesCoffee,servesDessert,"
                    "servesLunch,servesDinner,googleMapsUri"
                ),
            )

            location = details.get("location", {})
            distance_miles = None
            if location.get("latitude") is not None and location.get("longitude") is not None:
                distance_miles = round(
                    haversine_miles(
                        self.state["user_location"]["lat"],
                        self.state["user_location"]["lng"],
                        location["latitude"],
                        location["longitude"],
                    ),
                    2,
                )

            review_stats = summarize_recent_reviews(details.get("reviews", []))
            recency_weighted_rating = review_stats.get("recency_weighted_rating")
            estimated_travel_minutes = estimate_travel_minutes(distance_miles, self.state.get("travel_mode"))
            next_close_time = details.get("currentOpeningHours", {}).get("nextCloseTime")

            if self.state.get("when") == "now" and will_close_before_arrival(next_close_time, estimated_travel_minutes):
                continue

            score = 0.0
            if recency_weighted_rating is not None:
                score += recency_weighted_rating * 2.5
            else:
                score += float(details.get("rating", 0) or 0) * 1.5
            score += min(review_stats.get("recent_review_count", 0), 20) / 5
            score += min(details.get("userRatingCount", 0), 4000) / 2000
            if distance_miles is not None:
                score += max(0.0, 4.0 - min(distance_miles, 4.0))
            if self.state.get("meal_type") == "coffee" and details.get("servesCoffee"):
                score += 1.0
            if self.state.get("meal_type") == "dessert" and details.get("servesDessert"):
                score += 1.0
            if self.state.get("place_type") == "takeout" and details.get("takeout"):
                score += 1.0

            place_record = {
                "name": details.get("displayName", {}).get("text"),
                "address": details.get("formattedAddress"),
                "rating": details.get("rating"),
                "user_rating_count": details.get("userRatingCount"),
                "distance_miles": distance_miles,
                "estimated_travel_minutes": estimated_travel_minutes,
                "lat": location.get("latitude"),
                "lng": location.get("longitude"),
                "primary_type": details.get("primaryTypeDisplayName", {}).get("text"),
                "price_level": details.get("priceLevel"),
                "open_now": details.get("currentOpeningHours", {}).get("openNow"),
                "next_close_time": next_close_time,
                "takeout": details.get("takeout"),
                "delivery": details.get("delivery"),
                "dine_in": details.get("dineIn"),
                "review_summary": details.get("reviewSummary", {}).get("text"),
                "editorial_summary": details.get("editorialSummary", {}).get("text"),
                "recent_review_count": review_stats.get("recent_review_count"),
                "recency_weighted_rating": recency_weighted_rating,
                "recent_reviews": review_stats.get("recent_reviews"),
                "google_maps_url": details.get("googleMapsUri"),
                "directions_url": build_directions_url(self.state["user_location"], {"lat": location.get("latitude"), "lng": location.get("longitude")}),
                "score": round(score, 2),
            }
            place_record["unmet_criteria"] = unmet_criteria_for_place(place_record, self.state)
            results.append(place_record)

        self.last_results = sorted(results, key=lambda item: item["score"], reverse=True)[:limit]
        return self.last_results

    def _final_answer_with_gemini(self, results: List[Dict[str, Any]]) -> str:
        prompt = f"""
You are a warm restaurant recommendation assistant.

Write the final answer as bullet points, not a paragraph.
Return up to 5 recommendations by default.
Use only the data provided.
If a place does not fully satisfy the user request, explicitly mention which criteria it does not follow.
Do not invent facts.
If ambience is not explicit in the place data, say it is inferred from the summaries.
Mention distance, estimated travel time, rating, and recent-review strength when available.
If a place has a recency_weighted_rating, prefer describing that over the overall rating.
End with one short bullet noting that the ranking uses reviews from the last 24 months, with stronger weight on the last 12 months and extra penalty for months 12 to 24.

User preference state:
{json.dumps({k: v for k, v in self.state.items() if k != 'user_location'}, ensure_ascii=False)}

Resolved location:
{json.dumps(self.state['user_location'], ensure_ascii=False)}

Travel preference:
{travel_preference_label(self.state)}

Candidate places:
{json.dumps(results[:5], ensure_ascii=False)}
"""
        response = gemini_client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        return response.text.strip()

    def handle_message(self, user_message: str) -> str:
        self._append("user", user_message)
        analysis = self._analyze_with_gemini()
        self._merge_state(analysis.get("state", {}))
        self._apply_local_food_inference()
        self._apply_local_travel_inference(user_message)
        self._apply_local_location_fallbacks(user_message)

        try:
            if self.state.get("location_mode") in {"current", "manual"} and not self.state.get("user_location"):
                self._resolve_location()
        except Exception as exc:
            if parse_coordinate_pair(user_message):
                self.state["location_mode"] = "manual"
                self.state["manual_location"] = user_message.strip()
                self.state["user_location"] = parse_coordinate_pair(user_message)
            else:
                assistant_message = f"I could not resolve that location yet: {exc}"
                self._append("assistant", assistant_message)
                return assistant_message

        has_required_slots = bool(
            self.state.get("when")
            and self.state.get("cuisine")
            and (self.state.get("user_location") or self.state.get("location_mode"))
            and (self.state.get("location_mode") != "manual" or self.state.get("manual_location"))
            and self.state.get("travel_minutes")
        )
        ready = bool(analysis.get("ready_for_search")) or has_required_slots
        if not ready:
            if not self.state.get("when"):
                assistant_message = analysis.get("next_question") or "Do you want to eat now or later?"
            elif not self.state.get("cuisine") and is_vague_food_request(self.state.get("dish")):
                assistant_message = analysis.get("next_question") or "What is your preference?"
            elif self.state.get("location_mode") == "manual" and not self.state.get("manual_location"):
                assistant_message = "Please tell me the location you want me to search near, like Central Park or Empire State Building."
            elif not self.state.get("location_mode"):
                assistant_message = analysis.get("next_question") or "What location are you thinking of? You can type a place name or coordinates like 40.754314, -73.977541."
            elif not self.state.get("cuisine"):
                assistant_message = analysis.get("next_question") or "What cuisine are you in the mood for?"
            elif not self.state.get("travel_minutes"):
                assistant_message = analysis.get("next_question") or "How long are you willing to travel, and is that by walk or by car?"
            else:
                assistant_message = analysis.get("next_question") or "How long are you willing to travel, and is that by walk or by car?"
            self.last_results = []
            self._append("assistant", assistant_message)
            return assistant_message

        results = self._fetch_recommendations(limit=5)
        if not results:
            assistant_message = "I could not find a strong live match with your current filters. Try lowering the rating a bit or broadening the cuisine or area."
            self.last_results = []
            self._append("assistant", assistant_message)
            return assistant_message

        assistant_message = self._final_answer_with_gemini(results)
        self._append("assistant", assistant_message)
        return assistant_message


In [ ]:
bot = GeminiPlacesBot()
print("Start chatting. Type quit to stop.")
print("After you get recommendations, run the next cell to show the map.")

while True:
    user_message = input("You: ").strip()
    if user_message.lower() in {"quit", "exit", "stop"}:
        print("Bot: See you next time.")
        break
    try:
        reply = bot.handle_message(user_message)
    except Exception as exc:
        reply = f"Sorry, I hit an error: {exc}"
        bot.last_results = []
    print(f"Bot: {reply}")


In [ ]:
if bot.state.get("user_location") and bot.last_results:
    display(build_results_map(bot.state["user_location"], bot.last_results))
else:
    print("No map to show yet. Chat with the bot first until you get recommendations.")


## Example prompts

- `I am at Central Park and I am very hungry right now, I crave pizza, especially Chicago pizza and Hawaiian pizza, with Italian ambience, and at least 4 rating.`
- `I want to eat a dosa.`
- `I want to have coffee for a date.`

If the first prompt is incomplete, Gemini should ask the next missing question instead of jumping straight to search.
